In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/bakar_qureshi/miniconda3/envs/ml-cuda12.6/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Setup Base Model

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

if DEVICE == "cuda":
    
    device = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(device)

    total_memory = torch.cuda.get_device_properties(device).total_memory
    allocated_memory = torch.cuda.memory_allocated(device)
    reserved_memory = torch.cuda.memory_reserved(device)
    free_memory = total_memory - reserved_memory
    
    print(f"Backend: CUDA")
    print(f"GPU: {gpu_name}")
    print(f"Total Memory:     {total_memory / 1e6:.2f} MB | {total_memory / 1e9:.2f} GB")
    print(f"Allocated Memory: {allocated_memory / 1e6:.2f} MB | {allocated_memory / 1e9:.2f} GB")
    print(f"Reserved Memory:  {reserved_memory / 1e6:.2f} MB | {reserved_memory / 1e9:.2f} GB")
    print(f"Free Memory:      {free_memory / 1e6:.2f} MB | {free_memory / 1e9:.2f} GB")

elif DEVICE == "mps":
    # Note: MPS doesn't expose detailed memory stats like CUDA.
    # Apple Silicon uses unified memory (shared between CPU and GPU).
    # You can check total system memory as a proxy.
    import subprocess
    total_memory = int(subprocess.check_output(["sysctl", "-n", "hw.memsize"]).strip())
    
    print(f"Backend: MPS")
    print(f"Device: Apple Silicon (Metal Performance Shaders)")
    print(f"Total System Memory (unified): {total_memory / 1e6:.2f} MB | {total_memory / 1e9:.2f} GB")
    
    # Verify MPS works with a quick tensor test
    x = torch.tensor([1.0, 2.0]).to("mps")
    print(f"MPS tensor test: {x.device}")

else:
    print("No GPU available (no CUDA or MPS backend found)")

Using device: cuda
Backend: CUDA
GPU: NVIDIA GeForce GTX 1060
Total Memory:     6442.32 MB | 6.44 GB
Allocated Memory: 0.00 MB | 0.00 GB
Reserved Memory:  0.00 MB | 0.00 GB
Free Memory:      6442.32 MB | 6.44 GB


## Load Model

In [3]:
MODEL_NAME = "google/gemma-3-270m-it"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Model using device: {model.device}")
print(f"Model dtype: {model.dtype}")

Loading weights: 100%|██████████| 236/236 [00:01<00:00, 221.13it/s]


Model using device: cuda:0
Model dtype: torch.bfloat16


In [4]:
tokenizer("hello my name is bakar")

{'input_ids': [2, 23391, 1041, 1463, 563, 143440], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [5]:
input_string = "Hello my name is bakar"

token_tensor = torch.tensor(tokenizer(input_string)["input_ids"]).unsqueeze(0).to(model.device)

print(f"Token tensor shape: {token_tensor.shape}")
print(f"Token tensor device: {token_tensor.device}")
print(f"Token tensor dtype: {token_tensor.dtype}")
print(f"Token tensor: {token_tensor}")

outputs = model(token_tensor)
outputs.keys()

Token tensor shape: torch.Size([1, 6])
Token tensor device: cuda:0
Token tensor dtype: torch.int64
Token tensor: tensor([[     2,   9259,   1041,   1463,    563, 143440]], device='cuda:0')


odict_keys(['logits', 'past_key_values'])

In [6]:
logits = outputs.logits
logits.shape

torch.Size([1, 6, 262144])

In [7]:
predicted_ids = logits.argmax(dim=-1)
print(f"Predicted token IDs: {predicted_ids}")
print(f"Predicted token IDs shape: {predicted_ids.shape}")

Predicted token IDs: tensor([[  1106, 236888,   4389,    563,    870, 236761]], device='cuda:0')
Predicted token IDs shape: torch.Size([1, 6])


In [8]:
predicted_tokens = tokenizer.convert_ids_to_tokens(predicted_ids[0])
predicted_text = tokenizer.decode(predicted_ids[0])

print(f"Predicted tokens: {predicted_tokens}")
print(f"Predicted text: {predicted_text}")

print(f"Original input string: {input_string}")

Predicted tokens: ['import', '!', '▁friend', '▁is', '▁[', '.']
Predicted text: import! friend is [.
Original input string: Hello my name is bakar


In [9]:
def get_model_num_params(model):
    """
    Returns the number of trainable, non-trainable and total parameters of a PyTorch model.
    """
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params
    return {"trainable_params": trainable_params,
            "non_trainable_params": non_trainable_params,
            "total_params": total_params}

# Get parameters of our fine-tuned model
model_params = get_model_num_params(model)
print(f"Trainable parameters: {model_params['trainable_params']:,}")
print(f"Non-trainable parameters: {model_params['non_trainable_params']:,}")
print(f"Total parameters: {model_params['total_params']:,}")

Trainable parameters: 268,098,176
Non-trainable parameters: 0
Total parameters: 268,098,176


# Getting a dataset

In [10]:
import random
import ast
from datasets import load_dataset
from transformers import pipeline

In [11]:
dataset = load_dataset("mrdbourke/FoodExtract-1k")

In [12]:

print(f"Number of samples in the dataset: {len(dataset['train'])}")

Number of samples in the dataset: 1420


In [13]:
def get_random_idx(dataset):
    return random.randint(0, len(dataset['train']) - 1)

def get_random_sample(dataset):

    random_idx = get_random_idx(dataset)
    random_sample = dataset['train'][random_idx]

    example_input = random_sample['sequence']
    example_output = random_sample['gpt-oss-120b-label']
    example_output_condensed = random_sample['gpt-oss-120b-label-condensed']



    print(f"[INFO] Input:\n{example_input}\n")
    print()
    print(f"[INFO] Example structured JSON output (what we want to turn our raw texts into):\n")
    print(ast.literal_eval(example_output)) # ast.literal_eval turns the string into JSON
    print()
    print(f"[INFO] Example output condensed (we'll train our model to predict the condensed output since it uses less tokens than JSON):\n")
    print(example_output_condensed)

    return random_sample

In [14]:
tags_dict = {'np': 'nutrition_panel',
 'il': 'ingredient_list',
 'me': 'menu',
 're': 'recipe',
 'fi': 'food_items',
 'di': 'drink_items',
 'fa': 'food_advertistment',
 'fp': 'food_packaging'}

## Create the conversation style dataset

In [18]:
def sample_to_conversation_per_item(sample):
    """Helper function to convert an input sample to prompt-completion style."""
    return {
        "prompt": [
            {"role": "user", "content": sample["sequence"]} # Load the sequence from the dataset
        ],
        "completion": [
            {"role": "assistant", "content": sample["gpt-oss-120b-label-condensed"]} # Load the gpt-oss-120b generated label
        ]
    }

def sample_to_conversation_batch(batch):
    prompts = []
    completion = []

    for i in range(0, len(batch['sequence'])):
        # print(i)

        prompts.append(
                [{"role": "user", "content": batch["sequence"][i]}] # Load the sequence from the dataset
        )
         
        completion.append(
                [{"role": "assistant", "content": batch["gpt-oss-120b-label-condensed"][i]}] # Load the gpt-oss-120b generated label
        )

    return {
        "prompt": prompts,
        "completion": completion
    }

# sample_to_conversation_per_item(random_sample)
# sample_to_conversation_batch(dataset)

In [44]:
dataset_mapped_item = dataset.map(sample_to_conversation_per_item,
                      batched=False)

dataset_mapped_batch = dataset.map(sample_to_conversation_batch,
                      batched=True)

Map: 100%|██████████| 284/284 [00:00<00:00, 13499.96 examples/s]


In [ ]:
item_mapped_sample = get_random_sample(dataset_mapped_item)

[INFO] Input:
Top‑down view of a meticulously arranged food tableau: a slab of raw beef cheeks lying on a white marble board, its deep mahogany flesh glistening with faint pink marrow; a crystal‑clear glass carafe of cranberry juice beside it, the liquid a vivid ruby hue that catches the light; a bowl of golden cereal cornflakes spilling in a neat pile, each flake crisp and airy with a buttery sheen; a translucent tray of gelatin cubes, pale amber with a slight wobble, dusted with a fine powder of sugar; a bunch of fresh leeks sliced crosswise, the white stalks and pale‑green tops fanning out in concentric circles; a rack of raw pork spare ribs displayed skin‑side up, the pale pink meat and ivory fat marbled with subtle veins; a small porcelain ramekin of dark, glossy sauce hp, its surface glossy and speckled with tiny herb flecks; a mound of bright yellow rice, each grain separate and fluffy, radiating a buttery glow; a few sprigs of vivid green basil leaves, their glossy veins and ar

In [51]:

batch_mapped_sample = get_random_sample(dataset_mapped_batch)

[INFO] Input:
White ceramic bowl of Pepperoni Pizza Mac & Cheese with fresh basil and thyme, and red and white gingham kitchen towel BoulderLocavore.com


[INFO] Example structured JSON output (what we want to turn our raw texts into):

{'is_food_or_drink': True, 'tags': ['fi'], 'food_items': ['Pepperoni Pizza Mac & Cheese', 'basil', 'thyme'], 'drink_items': []}

[INFO] Example output condensed (we'll train our model to predict the condensed output since it uses less tokens than JSON):

food_or_drink: 1
tags: fi
foods: Pepperoni Pizza Mac & Cheese, basil, thyme
drinks:


In [54]:
item_mapped_sample

{'sequence': 'Top‑down view of a meticulously arranged food tableau: a slab of raw beef cheeks lying on a white marble board, its deep mahogany flesh glistening with faint pink marrow; a crystal‑clear glass carafe of cranberry juice beside it, the liquid a vivid ruby hue that catches the light; a bowl of golden cereal cornflakes spilling in a neat pile, each flake crisp and airy with a buttery sheen; a translucent tray of gelatin cubes, pale amber with a slight wobble, dusted with a fine powder of sugar; a bunch of fresh leeks sliced crosswise, the white stalks and pale‑green tops fanning out in concentric circles; a rack of raw pork spare ribs displayed skin‑side up, the pale pink meat and ivory fat marbled with subtle veins; a small porcelain ramekin of dark, glossy sauce hp, its surface glossy and speckled with tiny herb flecks; a mound of bright yellow rice, each grain separate and fluffy, radiating a buttery glow; a few sprigs of vivid green basil leaves, their glossy veins and ar

In [53]:
batch_mapped_sample

{'sequence': 'White ceramic bowl of Pepperoni Pizza Mac & Cheese with fresh basil and thyme, and red and white gingham kitchen towel BoulderLocavore.com',
 'image_url': 'https://boulderlocavore.com/wp-content/uploads/2017/08/Pepperoni-Pizza-Mac-and-Cheese-BoulderLocavore.com--272x408.jpg',
 'class_label': 'food',
 'source': 'coyo700m_dataset',
 'char_len': 138.0,
 'word_count': 22.0,
 'syn_or_real': 'real',
 'uuid': '1efe851d-4756-43a4-88bf-3b290b2dc1c7',
 'gpt-oss-120b-label': "{'is_food_or_drink': True, 'tags': ['fi'], 'food_items': ['Pepperoni Pizza Mac & Cheese', 'basil', 'thyme'], 'drink_items': []}",
 'gpt-oss-120b-label-condensed': 'food_or_drink: 1\ntags: fi\nfoods: Pepperoni Pizza Mac & Cheese, basil, thyme\ndrinks:',
 'target_food_names_to_use': None,
 'caption_detail_level': None,
 'num_foods': None,
 'target_image_point_of_view': None,
 'prompt': [{'role': 'user',
   'content': 'White ceramic bowl of Pepperoni Pizza Mac & Cheese with fresh basil and thyme, and red and white

#### Train / Test split

In [21]:
dataset = dataset['train'].train_test_split(test_size=0.2,
                                       shuffle=False,
                                       seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['sequence', 'image_url', 'class_label', 'source', 'char_len', 'word_count', 'syn_or_real', 'uuid', 'gpt-oss-120b-label', 'gpt-oss-120b-label-condensed', 'target_food_names_to_use', 'caption_detail_level', 'num_foods', 'target_image_point_of_view', 'prompt', 'completion'],
        num_rows: 1136
    })
    test: Dataset({
        features: ['sequence', 'image_url', 'class_label', 'source', 'char_len', 'word_count', 'syn_or_real', 'uuid', 'gpt-oss-120b-label', 'gpt-oss-120b-label-condensed', 'target_food_names_to_use', 'caption_detail_level', 'num_foods', 'target_image_point_of_view', 'prompt', 'completion'],
        num_rows: 284
    })
})

## Try Model

In [22]:
def create_easy_sample(input):

    template = {"role": "user", "content": input}
    return template

create_easy_sample("This is a test input to see if the function works.")

{'role': 'user',
 'content': 'This is a test input to see if the function works.'}

In [23]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer)

input_text = "Write me a poem about the winter nights"

easy_sample = create_easy_sample(input_text)
input_prompt = pipe.tokenizer.apply_chat_template([easy_sample],
                                                  tokenize=False,
                                                  add_generation_prompt=True)

default_outputs = pipe(input_prompt,
                       max_new_tokens=500,
                       disable_compile=True)

[transformers] Passing `generation_config` together with generation-related arguments=({'disable_compile', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [24]:
print(default_outputs[0]['generated_text'])

<bos><start_of_turn>user
Write me a poem about the winter nights<end_of_turn>
<start_of_turn>model
The world is hushed, a blanket white,
A silent hush, a gentle blight.
The frosted breath of winter's sigh,
Whispers secrets in the sky.

A silver moon, a pale, watchful eye,
Reflects the fading, fragile sky.
The trees stand stark, their branches bare,
A skeletal, skeletal prayer.

The snow descends, a chilling grace,
A silent, melancholic space.
Each frosted leaf, a fragile gleam,
Lost in the darkness, a waking dream.

The world below, a frosted scene,
Where frozen dreams are evergreen.
A quiet peace, a solace deep,
Where winter's icy grip is asleep.

And in this stillness, peace resides,
Awaiting spring's gentle tides.
The air is crisp, a whispered art,
A promise held within the heart.



In [25]:
print(tokenizer.chat_template)

{{ bos_token }}
{%- if messages[0]['role'] == 'system' -%}
    {%- if messages[0]['content'] is string -%}
        {%- set first_user_prefix = messages[0]['content'] + '

' -%}
    {%- else -%}
        {%- set first_user_prefix = messages[0]['content'][0]['text'] + '

' -%}
    {%- endif -%}
    {%- set loop_messages = messages[1:] -%}
{%- else -%}
    {%- set first_user_prefix = "" -%}
    {%- set loop_messages = messages -%}
{%- endif -%}
{%- for message in loop_messages -%}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) -%}
        {{ raise_exception("Conversation roles must alternate user/assistant/user/assistant/...") }}
    {%- endif -%}
    {%- if (message['role'] == 'assistant') -%}
        {%- set role = "model" -%}
    {%- else -%}
        {%- set role = message['role'] -%}
    {%- endif -%}
    {{ '<start_of_turn>' + role + '
' + (first_user_prefix if loop.first else "") }}
    {%- if message['content'] is string -%}
        {{ message['content'] | trim }}


## Let's try model on our sample

In [26]:
random_sample = get_random_sample(dataset)

[INFO] Input:
quay seafood and steak, bedford, restaurant review the beauty type, lifestyle blog northamptonshire


[INFO] Example structured JSON output (what we want to turn our raw texts into):

{'is_food_or_drink': True, 'tags': ['fi'], 'food_items': ['seafood', 'steak'], 'drink_items': []}

[INFO] Example output condensed (we'll train our model to predict the condensed output since it uses less tokens than JSON):

food_or_drink: 1
tags: fi
foods: seafood, steak
drinks:


In [30]:
random_sample["prompt"][0]["content"], random_sample["completion"]

('quay seafood and steak, bedford, restaurant review the beauty type, lifestyle blog northamptonshire',
 [{'role': 'assistant',
   'content': 'food_or_drink: 1\ntags: fi\nfoods: seafood, steak\ndrinks:'}])

In [31]:
input_prompt = pipe.tokenizer.apply_chat_template([random_sample["prompt"]],
                                   tokenize=False,
                                    add_generation_prompt=True)

default_outputs = pipe(input_prompt,
                       max_new_tokens=200,
                       disable_compile=True)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [41]:
print(default_outputs[0][0]['generated_text'])

<bos><start_of_turn>user
quay seafood and steak, bedford, restaurant review the beauty type, lifestyle blog northamptonshire<end_of_turn>
<start_of_turn>model
Okay, here's a review of a restaurant in Northamptonshire, focusing on the beauty and lifestyle aspects.



## Prompt the model

In [27]:
prompt_instruction = """Given the following target input text from an image caption, please extract the food and drink items to a list. 
If there are no food or drink items, return an empty list.

Return in the following format:
food_items: [food_item_1, food_item_2, food_item_3]
drink_items: [drink_item_4, drink_item_5]

For example:
Input text: Hello my name is Daniel.
Output:
food_items: []
drink_items: []

Example 2:
Input text: A plate of rice cakes, salmon, cottage cheese and small cherry tomatoes with a cup of tea.
Output:
food_items: ['rice cakes', 'salmon', 'cottage cheese', 'cherry tomatoes']
drink_items: ['cup of tea']

Return only the formatted output and nothing else.

Target input text: <targ_input_text>"""

def update_input_message_content(input_text):

    original_prompt = input_text["prompt"]["content"]
    new_prompt = prompt_instruction.replace("<targ_input_text>", original_prompt)

    new_input_text = {"role": "user", "content": new_prompt}

    return new_input_text

In [28]:
updated_input_text = update_input_message_content(random_sample)

input_prompt = pipe.tokenizer.apply_chat_template([updated_input_text],
                                   tokenize=False,
                                    add_generation_prompt=True)

default_outputs = pipe(input_prompt,
                       max_new_tokens=200,
                       max_length=200)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [29]:
print(default_outputs[0]['generated_text'])

<bos><start_of_turn>user
Given the following target input text from an image caption, please extract the food and drink items to a list. 
If there are no food or drink items, return an empty list.

Return in the following format:
food_items: [food_item_1, food_item_2, food_item_3]
drink_items: [drink_item_4, drink_item_5]

For example:
Input text: Hello my name is Daniel.
Output:
food_items: []
drink_items: []

Example 2:
Input text: A plate of rice cakes, salmon, cottage cheese and small cherry tomatoes with a cup of tea.
Output:
food_items: ['rice cakes', 'salmon', 'cottage cheese', 'cherry tomatoes']
drink_items: ['cup of tea']

Return only the formatted output and nothing else.

Target input text: Cleaver's Organic Beef Meatballs, 380g, Best Before 19-Sep-25. Ingredients: Organic Beef (83%), Water, Organic Onion, Organic Rice Flour, Organic Tapioca Starch, Salt, Organic Spices, Organic Sugar, Fruit and Spice Extracts, Organic Garlic, Fermented Rice. Ethical Pillars: Grass Fed, High

# Fine Tune Model

## Setup SFT Config

In [36]:
from trl import SFTConfig
import os

In [ ]:
torch_dtype = model.dtype

CHECKPOINT_DIR = "./checkpoints_models"
BASE_LEARNING_RATE = 5e-5
BATCH_SIZE = 4

print(f"[INFO] Using dtype: {torch_dtype}")
print(f"[INFO] Checkpoint directory: {CHECKPOINT_DIR}")
print(f"[INFO] Base learning rate: {BASE_LEARNING_RATE}")
print(f"[INFO] Batch size: {BATCH_SIZE}")


sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    max_length=512,
    packing=False,
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    completion_only_loss=True,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=BASE_LEARNING_RATE,
    fp16=(torch_dtype == torch.float16),
    bf16=(torch_dtype == torch.bfloat16),
    load_best_model_at_end=True,
    metric_for_best_model="mean_token_accuracy",
    greater_is_better=True,
    lr_scheduler_type="constant",
    push_to_hub=False,
    report_to="none"
)

SyntaxError: invalid syntax (3598827546.py, line 1)